# Model Retention Walk-Forward Gate

Purpose: audit replayable SOXL models against the current retention target: complete walk-forward evidence, strict max drawdown below 20%, and YTD return aligned to both the converted SOXL backtest reference and current best replay model.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'AGENTS.md').exists():
    ROOT = next(parent for parent in Path.cwd().parents if (parent / 'AGENTS.md').exists())
CARD = ROOT / 'runtime/model_selection/card/model_retention_walkforward_gate_v1'
CARD.mkdir(parents=True, exist_ok=True)
ROOT, CARD

(PosixPath('/Users/rziiiii/dev/workspace'),
 PosixPath('/Users/rziiiii/dev/workspace/runtime/model_selection/card/model_retention_walkforward_gate_v1'))

In [2]:
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.model_retention_gate import scan_model_retention_gates, write_retention_gate_outputs

result = scan_model_retention_gates(root=ROOT, max_walkforward_drawdown_pct=20.0, best_return_ratio=0.95)
write_retention_gate_outputs(result, output_dir=CARD)
summary = result['summary']
summary

{'generated_at': '2026-06-15T23:34:05.158922+00:00',
 'profiles_scanned': 12,
 'passing_profiles': [],
 'passing_profile_count': 0,
 'reference_profile': 'soxl_backtest_policy_reimplementation_v1',
 'reference_return_pct': 146.043798,
 'reference_max_drawdown_pct': 13.554089,
 'current_best_return_pct': 224.361539,
 'best_return_ratio': 0.95,
 'required_best_return_pct': 213.143462,
 'max_walkforward_drawdown_pct': 20.0,
 'required_wf_windows': ['wf_2020', 'wf_2022', 'wf_2023_2024'],
 'closest_wf_drawdown_candidate': {'profile': 'mode_conditioned_rebound_reward_train_v2',
  'worst_wf_max_drawdown_pct': 21.83172,
  'wf_drawdown_gap_pct': 1.83172,
  'missing_wf_window_count': 0,
  'ytd_return_pct': 177.573569,
  'best_return_gap_pct': 35.569893},
 'closest_ytd_return_candidate': {'profile': 'smoke',
  'ytd_return_pct': 224.361539,
  'best_return_gap_pct': 0.0,
  'worst_wf_max_drawdown_pct': None,
  'wf_drawdown_gap_pct': None,
  'missing_wf_window_count': 3}}

In [3]:
rows = pd.DataFrame(result['rows'])
display_columns = [
    'profile', 'status', 'ytd_return_pct', 'ytd_max_drawdown_pct',
    'best_return_gap_pct', 'soxlbacktest_return_gap_pct',
    'wf_windows_present', 'missing_wf_window_count',
    'worst_wf_max_drawdown_pct', 'wf_drawdown_gap_pct',
    'gate_pass', 'retention_decision', 'failures',
]
display(rows[display_columns])


,profile,status,ytd_return_pct,ytd_max_drawdown_pct,best_return_gap_pct,soxlbacktest_return_gap_pct,wf_windows_present,missing_wf_window_count,worst_wf_max_drawdown_pct,wf_drawdown_gap_pct,gate_pass,retention_decision,failures
0,soxl_backtest_policy_reimplementation_v1,reference,146.043798,13.554089,NaN,NaN,,0,NaN,NaN,True,keep_reference_baseline,[]
1,balanced,archived,159.205721,29.698281,53.937740,0.000000,,3,NaN,NaN,False,retain_as_evidence_only,"[missing_wf_window:wf_2020, missing_wf_window:..."
2,balanced_macro_v1,candidate,196.749008,29.901822,16.394453,0.000000,,3,NaN,NaN,False,retain_as_evidence_only,"[missing_wf_window:wf_2020, missing_wf_window:..."
3,balanced_rule_residual_v19,archived,173.299227,24.399161,39.844235,0.000000,"wf_2020,wf_2022,wf_2023_2024",0,46.653957,26.653957,False,retain_as_evidence_only,"[wf_max_drawdown_ge_20:wf_2020, wf_max_drawdow..."
4,smoke,canonical,224.361539,30.346650,0.000000,0.000000,,3,NaN,NaN,False,retain_as_evidence_only,"[missing_wf_window:wf_2020, missing_wf_window:..."
5,balanced_rule_residual_v22,canonical,116.882644,26.278174,96.260818,29.161154,"wf_2020,wf_2022,wf_2023_2024",0,54.665610,34.665610,False,retain_as_evidence_only,"[wf_max_drawdown_ge_20:wf_2020, wf_max_drawdow..."
6,balanced_rule_residual_imitation_v4,candidate,145.031235,25.999446,68.112227,1.012563,"wf_2020,wf_2022,wf_2023_2024",0,35.710864,15.710864,False,retain_as_evidence_only,"[wf_max_drawdown_ge_20:wf_2020, wf_max_drawdow..."
7,drawdown_rebound_reward_train_v1,ablation,172.832548,27.310377,40.310914,0.000000,"wf_2020,wf_2022,wf_2023_2024",0,23.546830,3.546830,False,retain_as_evidence_only,"[wf_max_drawdown_ge_20:wf_2020, ytd_return_bel..."
8,mode_conditioned_rebound_reward_train_v2,ablation,177.573569,27.278641,35.569893,0.000000,"wf_2020,wf_2022,wf_2023_2024",0,21.831720,1.831720,False,retain_as_evidence_only,"[wf_max_drawdown_ge_20:wf_2020, wf_max_drawdow..."
9,state_conditioned_drawdown_rebound_reward_trai...,ablation,196.505054,31.390931,16.638408,0.000000,"wf_2020,wf_2022,wf_2023_2024",0,27.608986,7.608986,False,retain_as_evidence_only,"[wf_max_drawdown_ge_20:wf_2020, ytd_return_bel..."


In [4]:
candidate_rows = rows[rows['profile'] != 'soxl_backtest_policy_reimplementation_v1'].copy()
diagnostic_candidates = candidate_rows.sort_values(
    ['missing_wf_window_count', 'wf_drawdown_gap_pct', 'best_return_gap_pct'],
    na_position='last',
)[[
    'profile', 'status', 'ytd_return_pct', 'best_return_gap_pct',
    'worst_wf_max_drawdown_pct', 'wf_drawdown_gap_pct',
    'missing_wf_window_count', 'failures',
]]
diagnostic_candidates.to_csv(CARD / 'diagnostic_candidates.csv', index=False)
retrain_handoff = {
    'decision': 'train_state_conditioned_multi_objective_no_promotion',
    'promotion': 'blocked',
    'reference_baseline': summary['reference_profile'],
    'reference_return_pct': summary['reference_return_pct'],
    'required_best_return_pct': summary['required_best_return_pct'],
    'max_walkforward_drawdown_pct': summary['max_walkforward_drawdown_pct'],
    'required_wf_windows': summary['required_wf_windows'],
    'closest_wf_drawdown_candidate': summary.get('closest_wf_drawdown_candidate'),
    'closest_ytd_return_candidate': summary.get('closest_ytd_return_candidate'),
    'objective_terms': [
        'walkforward_drawdown_gap_penalty',
        'walkforward_relative_return_gap_penalty',
        'state_conditioned_downtrend_drawdown_cap',
        'rebound_participation_floor_without_exceeding_wf20dd',
    ],
    'stop_conditions': [
        'do_not_promote_missing_walkforward_windows',
        'do_not_promote_worst_wf_maxdd_ge_20',
        'do_not_promote_ytd_return_below_best_alignment',
    ],
}
(CARD / 'retrain_handoff.json').write_text(json.dumps(retrain_handoff, indent=2, sort_keys=True) + '\n', encoding='utf-8')
display(pd.DataFrame([
    retrain_handoff['closest_wf_drawdown_candidate'],
    retrain_handoff['closest_ytd_return_candidate'],
]))


,profile,worst_wf_max_drawdown_pct,wf_drawdown_gap_pct,missing_wf_window_count,ytd_return_pct,best_return_gap_pct
0,mode_conditioned_rebound_reward_train_v2,21.83172,1.83172,0,177.573569,35.569893
1,smoke,NaN,NaN,3,224.361539,0.000000


## Conclusion

- `soxl_backtest_policy_reimplementation_v1` is retained as the converted Qlib-compatible rule-policy reference baseline. It remains a baseline, not a runnable PPO actor.
- No replayable RL/model profile currently satisfies all gates: complete `wf_2020`, `wf_2022`, `wf_2023_2024`; every WF MaxDD `<20%`; YTD return `>=138.644977%`; and YTD return `>=95%` of the current best replay (`smoke`, `224.361539%`).
- The closest complete-WF drawdown candidate should be used as the risk-shaping reference; the highest-return candidate should be used only as a participation/return reference if its walk-forward windows are missing.
- This card writes `diagnostic_candidates.csv` and `retrain_handoff.json` so the next training branch has explicit objective terms instead of promoting dashboard-visible evidence-only models.
- Next action: train/select a state-conditioned multi-objective candidate and rerun this same gate; do not keep a final model unless this card reports `passing_profile_count > 0`.
